# Week 6 — EDA II: relationships + hypothesis drafting

### Deliverable
- 4 relationship plots + 2 hypotheses in H0/H1 form


In [ ]:
# Core imports (kept minimal for beginners)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

# Dataset URL (UCI Heart Disease - Cleveland)
UCI_URL = "https://www.archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"

# Column names for processed.cleveland.data (14 columns commonly used in teaching)
COLS = ["age","sex","cp","trestbps","chol","fbs","restecg","thalach",
        "exang","oldpeak","slope","ca","thal","num"]


In [ ]:
import ssl
import io
import urllib.request # Added this import

def load_raw():
    # Create an unverified SSL context to bypass certificate verification
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE

    # Open the URL with the unverified context
    with urllib.request.urlopen(UCI_URL, context=ctx) as url_response:
        # Read the content and decode it
        s = url_response.read().decode('utf-8')
    
    # Use io.StringIO to make the string behave like a file for pandas.read_csv
    df_raw = pd.read_csv(io.StringIO(s), header=None, names=COLS)
    return df_raw

def coerce_types(df_raw):
    # Missing values are sometimes encoded as "?"
    df = df_raw.replace("?", np.nan).copy()

    # Convert numeric-looking columns
    numeric_cols = ["age","trestbps","chol","thalach","oldpeak","ca","thal","num"]
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Binary target: disease present if num > 0 (UCI convention)
    df["disease"] = (df["num"] > 0).astype("Int64")
    return df

df_raw = load_raw()
df = coerce_types(df_raw)

df.head()


In [ ]:
def clean_heart(df_raw):
    df = df_raw.replace("?", np.nan).copy()
    numeric_cols = ["age","trestbps","chol","thalach","oldpeak","ca","thal","num"]
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df["disease"] = (df["num"] > 0).astype("Int64")
    return df.dropna(subset=["disease"])

df_clean = clean_heart(df_raw)


## Plot A — Numeric vs disease


In [ ]:
col = "thalach"  # TODO
g0 = df_clean.loc[df_clean["disease"]==0, col].dropna()
g1 = df_clean.loc[df_clean["disease"]==1, col].dropna()

plt.figure()
plt.boxplot([g0, g1], labels=["no disease","disease"])
plt.ylabel(col)
plt.title(f"{col} by disease group")
plt.show()

print("Counts:", len(g0), len(g1))


## Plot B — Categorical vs disease


In [ ]:
cat = "cp"  # TODO
ct = pd.crosstab(df_clean[cat], df_clean["disease"], dropna=False)
display(ct)

props = ct.div(ct.sum(axis=1), axis=0)
plt.figure()
props.plot(kind="bar", stacked=True)
plt.ylabel("proportion")
plt.title(f"Disease proportion by {cat}")
plt.show()


## Plot C — Numeric vs numeric (colored by disease)


In [ ]:
xcol, ycol = "age", "thalach"  # TODO
d0 = df_clean[df_clean["disease"]==0]
d1 = df_clean[df_clean["disease"]==1]

plt.figure()
plt.scatter(d0[xcol], d0[ycol], alpha=0.6, label="no disease")
plt.scatter(d1[xcol], d1[ycol], alpha=0.6, label="disease")
plt.xlabel(xcol); plt.ylabel(ycol)
plt.title(f"{ycol} vs {xcol} by disease")
plt.legend()
plt.show()


## Plot D — Stratified view (probe confounding)


In [ ]:
col = "thalach"
for sex_code in sorted(df_clean["sex"].dropna().unique()):
    sub = df_clean[df_clean["sex"]==sex_code]
    g0 = sub.loc[sub["disease"]==0, col].dropna()
    g1 = sub.loc[sub["disease"]==1, col].dropna()

    plt.figure()
    plt.boxplot([g0, g1], labels=["no disease","disease"])
    plt.ylabel(col)
    plt.title(f"{col} by disease (sex={sex_code})")
    plt.show()

    print("sex=", sex_code, "counts:", len(g0), len(g1))


## TODO — Two hypotheses (H0/H1)
Hypothesis 1:
- H0:
- H1:
- Planned test:

Hypothesis 2:
- H0:
- H1:
- Planned test:
